# 第14・15回：CNNとLLMの基礎 — MNIST画像分類実習

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session14_15_cnn_llm_mnist.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

> この教材では、質問文の「LMM」を **LLM（Large Language Model：大規模言語モデル）** の意味として扱います。

画像を扱うCNNと、文章を扱うLLMは一見まったく異なるモデルです。しかし、どちらも「データを数値に変換し、重要な特徴を学習し、誤差が小さくなるようにパラメータを更新する」という深層学習の考え方を共有しています。

第14回ではパーセプトロン、活性化関数、CNNの仕組みを整理します。第15回ではLLMの基礎を学んだ後、MNISTを用いてCNNを構築・学習・評価します。

---

## 📋 到達目標

1. パーセプトロンと多層パーセプトロンの違いを説明できる
2. 活性化関数が非線形な分類に必要な理由を説明できる
3. バックプロパゲーションとオプティマイザの役割を区別できる
4. Dropoutの目的とdrop rateの意味を説明できる
5. 畳み込みとプーリングの役割を説明できる
6. CNNが画像認識に適している理由を説明できる
7. トークン、埋め込み、Self-Attention、次トークン予測を説明できる
8. CNNとLLMの共通点・相違点を整理できる
9. KerasでMNIST分類用CNNを実装し、学習曲線や混同行列から評価できる

---

# 第14回：パーセプトロン・活性化関数・CNNの基礎

## 1. 深層学習に共通する考え方

### 1-1. パーセプトロン：人工ニューロンの基本単位

パーセプトロンは、複数の入力を受け取り、1つの出力を返す最も基本的な人工ニューロンです。それぞれの入力に重みを掛けて合計し、バイアスを加えた後、活性化関数へ渡します。

```mermaid
flowchart LR
    x1(["入力 x₁"]):::input
    x2(["入力 x₂"]):::input
    x3(["入力 x₃"]):::input
    bias(["バイアス b"]):::bias

    sum["加重和<br/>z = Σ wᵢxᵢ + b"]:::process
    activation["活性化関数<br/>a = f(z)"]:::activation
    output(["出力 ŷ"]):::output

    x1 -->|"重み w₁"| sum
    x2 -->|"重み w₂"| sum
    x3 -->|"重み w₃"| sum
    bias -.-> sum
    sum --> activation --> output

    classDef input fill:#dbeafe,stroke:#2563eb,color:#172554,stroke-width:2px
    classDef bias fill:#fef3c7,stroke:#d97706,color:#451a03,stroke-width:2px
    classDef process fill:#ede9fe,stroke:#7c3aed,color:#2e1065,stroke-width:2px
    classDef activation fill:#dcfce7,stroke:#16a34a,color:#052e16,stroke-width:2px
    classDef output fill:#fee2e2,stroke:#dc2626,color:#450a0a,stroke-width:2px
```

計算は次式で表せます。

$$
z = \sum_i w_i x_i + b, \qquad a = f(z)
$$

- $x_i$：入力
- $w_i$：入力の重要度を表す重み
- $b$：バイアス
- $f$：ReLUなどの活性化関数
- $a$：次の層に渡す出力

初期のパーセプトロンでは、加重和が基準以上なら1、未満なら0を返すステップ関数が使われました。これは直線や平面で分けられる問題には対応できますが、XORのように1本の直線では分けられない問題を学習できません。

### 1-2. 多層パーセプトロン（MLP）

複数のパーセプトロンを層として並べたものが**多層パーセプトロン（Multi-Layer Perceptron: MLP）**です。

```mermaid
flowchart LR
    subgraph inputLayer["入力層"]
        direction TB
        x1(("x₁")):::input
        x2(("x₂")):::input
    end

    subgraph hiddenLayer1["隠れ層1"]
        direction TB
        h1(("h₁")):::hidden
        h2(("h₂")):::hidden
        h3(("h₃")):::hidden
    end

    subgraph hiddenLayer2["隠れ層2"]
        direction TB
        h4(("h₄")):::hidden
        h5(("h₅")):::hidden
    end

    subgraph outputLayer["出力層"]
        direction TB
        y1(("ŷ₁")):::output
        y2(("ŷ₂")):::output
    end

    x1 --> h1
    x1 --> h2
    x1 --> h3
    x2 --> h1
    x2 --> h2
    x2 --> h3

    h1 --> h4
    h1 --> h5
    h2 --> h4
    h2 --> h5
    h3 --> h4
    h3 --> h5

    h4 --> y1
    h4 --> y2
    h5 --> y1
    h5 --> y2

    classDef input fill:#dbeafe,stroke:#2563eb,color:#172554,stroke-width:2px
    classDef hidden fill:#dcfce7,stroke:#16a34a,color:#052e16,stroke-width:2px
    classDef output fill:#fee2e2,stroke:#dc2626,color:#450a0a,stroke-width:2px
```

各矢印には学習によって更新される重みがあります。各隠れユニットでは「加重和 → 活性化関数」を計算し、その出力を次の層へ渡します。

- **入力層**：元データを受け取る
- **隠れ層**：入力を段階的に変換し、分類に役立つ特徴を作る
- **出力層**：目的に合わせた予測値やクラス確率を返す

`Dense` は、前の層のすべてのユニットと接続する全結合層です。層を重ねることで複雑な表現を作れますが、そのためには各層に活性化関数を入れる必要があります。

### 1-3. 活性化関数はなぜ必要か

活性化関数は、加重和を次の層へどのように伝えるかを決める関数です。ニューロンが「どの入力に、どの程度反応するか」を表します。

活性化関数を使わず、線形変換だけを何層重ねても、全体は結局1回の線形変換と同じです。

$$
W_2(W_1x + b_1) + b_2 = (W_2W_1)x + (W_2b_1 + b_2)
$$

そこで、層の間に**非線形な活性化関数**を入れます。これにより、曲線や複雑な領域でクラスを分けられるようになります。

| 活性化関数 | 出力範囲 | 主な用途・特徴 |
| --- | --- | --- |
| ステップ関数 | 0または1 | 初期のパーセプトロン。微分できないため、現在の深層学習の学習には通常使わない |
| Sigmoid | 0〜1 | 二値分類の出力層。値を確率として解釈しやすい |
| tanh | -1〜1 | 0を中心とした出力。古典的な隠れ層や一部の系列モデルで使用 |
| ReLU | 0以上 | 隠れ層で広く使用。計算が単純で勾配消失を緩和しやすい |
| Softmax | 各値が0〜1、合計1 | 多クラス分類の出力層。クラスごとの確率を表す |

#### ReLU

ReLU（Rectified Linear Unit）は、負の入力を0にし、正の入力をそのまま通します。

$$
\mathrm{ReLU}(x) = \max(0, x)
$$

今回のCNNでは、`Conv2D`層と`Dense(64)`層にReLUを使います。画像中の特徴に強く反応した正の値を残しながら、モデルに非線形性を加えます。

#### SigmoidとSoftmax

Sigmoidは1つの値を0〜1へ変換するため、「該当する／しない」のような二値分類の出力に向いています。Softmaxは複数の値を合計1の確率へ変換するため、0〜9を選ぶMNISTのような多クラス分類に向いています。

```text
二値分類：     Dense(1, activation="sigmoid")
10クラス分類：Dense(10, activation="softmax")
```

> 活性化関数は目的に合わせて選びます。特に出力層の活性化関数、正解ラベルの形式、損失関数の組み合わせが重要です。

### 1-4. 演習：パーセプトロンで論理ゲートを作る

最初に、パーセプトロンが直線で分類する様子を確認します。2つの入力 $x_1, x_2$ に対して、ステップ関数を使うパーセプトロンを実装します。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def step_function(z):
    """0以上なら1、0未満なら0を返すステップ関数。"""
    return (z >= 0).astype(int)

def perceptron_predict(X, weights, bias):
    """パーセプトロンの予測値と、活性化前の加重和を返す。"""
    z = X @ weights + bias
    return step_function(z), z


# x1, x2 の全組み合わせ
X_logic = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
])

# ANDゲート：x1とx2が両方1のときだけ1
and_pred, and_z = perceptron_predict(
    X_logic,
    weights=np.array([1.0, 1.0]),
    bias=-1.5,
)

# ORゲート：x1とx2のどちらかが1なら1
or_pred, or_z = perceptron_predict(
    X_logic,
    weights=np.array([1.0, 1.0]),
    bias=-0.5,
)

logic_results = pd.DataFrame({
    "x1": X_logic[:, 0],
    "x2": X_logic[:, 1],
    "ANDの加重和": and_z,
    "ANDの予測": and_pred,
    "ORの加重和": or_z,
    "ORの予測": or_pred,
})
logic_results


重みは同じですが、バイアスを変えると分類境界の位置が変わります。次の図では、実線がAND、破線がORの分類境界です。


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

ax.scatter(
    X_logic[:, 0], X_logic[:, 1],
    c=and_pred, cmap="coolwarm", s=180,
    edgecolor="black", zorder=3,
)

x_line = np.linspace(-0.25, 1.25, 200)
ax.plot(x_line, 1.5 - x_line, label="AND boundary", linewidth=2)
ax.plot(x_line, 0.5 - x_line, "--", label="OR boundary", linewidth=2)

for x1, x2 in X_logic:
    ax.text(x1 + 0.04, x2 + 0.04, f"({x1}, {x2})", fontsize=10)

ax.set(
    xlim=(-0.25, 1.25), ylim=(-0.25, 1.25),
    xlabel="x1", ylabel="x2",
    title="Decision boundaries of a perceptron",
)
ax.legend()
plt.show()


#### ミニ演習

1. ANDとORで、重みが同じでも結果が変わるのはなぜですか。
2. `bias=-1.5` を `-1.0` や `-2.0` に変えるとANDの結果はどうなりますか。
3. NOTゲートを作る重みとバイアスを考えてください。入力は1つで構いません。
4. XORでは `(0, 0)` と `(1, 1)` が0、`(0, 1)` と `(1, 0)` が1です。4点を1本の直線で分けられるか、図を描いて考えてください。

> 単層パーセプトロンはANDやORを表現できますが、XORを1本の直線で分類できません。XORには、隠れ層と非線形な活性化関数が必要です。

### 1-5. 演習：活性化関数の有無で分類境界を比較する

今度は、直線では分けにくい半月状のデータを使います。同じ数の層とユニットを持つ2つのモデルを作り、隠れ層の活性化関数だけを変えます。


In [ ]:
import tensorflow as tf
import keras
from keras import layers

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
keras.utils.set_random_seed(SEED)

X_moons, y_moons = make_moons(
    n_samples=1000,
    noise=0.20,
    random_state=SEED,
)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moons,
    y_moons,
    test_size=0.25,
    stratify=y_moons,
    random_state=SEED,
)

scaler_m = StandardScaler()
X_train_m = scaler_m.fit_transform(X_train_m)
X_test_m = scaler_m.transform(X_test_m)

plt.figure(figsize=(6, 5))
plt.scatter(
    X_train_m[:, 0], X_train_m[:, 1],
    c=y_train_m, cmap="coolwarm", s=20, alpha=0.8,
)
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("Nonlinear two-class data: make_moons")
plt.show()


#### モデルA：隠れ層に活性化関数を入れない


In [ ]:
def build_mlp(hidden_activation, model_name):
    model = keras.Sequential([
        layers.Input(shape=(2,)),
        layers.Dense(8, activation=hidden_activation),
        layers.Dense(8, activation=hidden_activation),
        layers.Dense(1, activation="sigmoid"),
    ], name=model_name)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.01),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


linear_model = build_mlp(
    hidden_activation=None,
    model_name="without_hidden_activation",
)

linear_history = linear_model.fit(
    X_train_m,
    y_train_m,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    verbose=0,
)

linear_loss, linear_accuracy = linear_model.evaluate(
    X_test_m, y_test_m, verbose=0
)
print(f"活性化なしのテスト正解率: {linear_accuracy:.3f}")


隠れ層を2層にしても、各層が線形変換だけなら、すべてをまとめて1回の線形変換として表せます。最後のSigmoidが0.5になる位置は、活性化前の値が0になる位置なので、分類境界は直線のままです。

#### モデルB：隠れ層にReLUを入れる


In [ ]:
keras.utils.set_random_seed(SEED)

relu_model = build_mlp(
    hidden_activation="relu",
    model_name="with_relu",
)

relu_history = relu_model.fit(
    X_train_m,
    y_train_m,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    verbose=0,
)

relu_loss, relu_accuracy = relu_model.evaluate(
    X_test_m, y_test_m, verbose=0
)
print(f"ReLUありのテスト正解率: {relu_accuracy:.3f}")

comparison_mlp = pd.DataFrame({
    "model": ["活性化なし", "ReLUあり"],
    "parameters": [linear_model.count_params(), relu_model.count_params()],
    "test_loss": [linear_loss, relu_loss],
    "test_accuracy": [linear_accuracy, relu_accuracy],
})
comparison_mlp


#### 分類境界を並べて確認する


In [ ]:
def plot_decision_boundary(ax, model, X, y, title):
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, 250),
        np.linspace(x2_min, x2_max, 250),
    )
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    probability = model.predict(grid, verbose=0).reshape(xx1.shape)

    ax.contourf(
        xx1, xx2, probability,
        levels=np.linspace(0, 1, 11), cmap="coolwarm", alpha=0.35,
    )
    ax.contour(xx1, xx2, probability, levels=[0.5], colors="black", linewidths=2)
    ax.scatter(
        X[:, 0], X[:, 1],
        c=y, cmap="coolwarm", s=20, edgecolor="white", linewidth=0.3,
    )
    ax.set(xlabel="feature 1", ylabel="feature 2", title=title)


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_decision_boundary(
    axes[0], linear_model, X_test_m, y_test_m,
    f"No hidden activation (accuracy={linear_accuracy:.3f})",
)
plot_decision_boundary(
    axes[1], relu_model, X_test_m, y_test_m,
    f"ReLU (accuracy={relu_accuracy:.3f})",
)
plt.tight_layout()
plt.show()


#### 比較演習

1. 左右の分類境界は、それぞれどのような形になりましたか。
2. 2つのモデルは層数とユニット数が同じなのに、なぜ結果が異なるのでしょうか。
3. ReLUを`tanh`へ変更すると、分類境界と正解率はどう変わりますか。
4. `Dense(8)`を`Dense(2)`へ変更すると、表現力はどう変化しますか。
5. 活性化なしのモデルで隠れ層をさらに増やしても、曲線の分類境界を作れない理由を数式と結び付けて説明してください。

> この演習の重要点は、単に「ReLUの方が高精度」ということではありません。**非線形な活性化関数が、MLPに曲がった分類境界を表現する能力を与える**ことを、図と結果から確認してください。

### 1-6. ニューラルネットワークはどう学習するか

学習は次の流れを繰り返します。

1. **順伝播**：入力から予測値を計算する
2. **損失の計算**：予測と正解のずれを数値化する
3. **誤差逆伝播**：各パラメータが損失に与えた影響を計算する
4. **最適化**：Adamなどを使って重みを更新する

```mermaid
flowchart LR
    input(["入力 X"]):::data
    model["モデル<br/>現在の重み W, b"]:::model
    prediction(["予測 ŷ"]):::data
    loss["損失 L(y, ŷ)<br/>正解との差"]:::loss
    backprop["バックプロパゲーション<br/>勾配を計算"]:::backprop
    optimizer["オプティマイザ<br/>重みを更新"]:::optimizer

    input -->|"順伝播"| model --> prediction
    prediction --> loss
    loss -->|"出力側から逆向き"| backprop
    backprop -->|"∂L/∂W, ∂L/∂b"| optimizer
    optimizer -.->|"更新した W, b"| model

    classDef data fill:#dbeafe,stroke:#2563eb,color:#172554,stroke-width:2px
    classDef model fill:#dcfce7,stroke:#16a34a,color:#052e16,stroke-width:2px
    classDef loss fill:#fee2e2,stroke:#dc2626,color:#450a0a,stroke-width:2px
    classDef backprop fill:#ede9fe,stroke:#7c3aed,color:#2e1065,stroke-width:2px
    classDef optimizer fill:#fef3c7,stroke:#d97706,color:#451a03,stroke-width:2px
```

#### バックプロパゲーションの役割

**バックプロパゲーション（誤差逆伝播法）**は、損失を小さくするには各重みをどちらへ、どの程度変えるべきかを調べる方法です。出力層で計算した損失の影響を、連鎖律を使って後ろの層から前の層へ伝えます。

1つの重み $w$ に対する勾配は、概念的には次のように計算されます。

$$
\frac{\partial L}{\partial w}
= \frac{\partial L}{\partial a}
  \frac{\partial a}{\partial z}
  \frac{\partial z}{\partial w}
$$

- 勾配が正：$w$を大きくすると損失が増える方向
- 勾配が負：$w$を大きくすると損失が減る方向
- 勾配の絶対値：その重みが損失に与える影響の大きさ

重要なのは、バックプロパゲーションとオプティマイザの役割を区別することです。

| 処理 | 役割 |
| --- | --- |
| バックプロパゲーション | 各パラメータについて損失の勾配を計算する |
| オプティマイザ | 計算された勾配を使ってパラメータを更新する |

単純な勾配降下法では、学習率を $\eta$ として次のように重みを更新します。

$$
w_{\mathrm{new}} = w_{\mathrm{old}} - \eta \frac{\partial L}{\partial w}
$$

学習率が小さすぎると学習に時間がかかり、大きすぎると最適な値を飛び越えて損失が安定しないことがあります。今回使用するAdamは、パラメータごとに更新幅を調整する代表的なオプティマイザです。

#### 確認問題

1. バックプロパゲーションとオプティマイザは、それぞれ何を担当しますか。
2. 勾配が正の場合、勾配降下法では重みを増やしますか、減らしますか。
3. 学習率が大きすぎる場合、損失はどのような変化をする可能性がありますか。
4. 活性化関数が微分可能、またはほぼすべての点で微分可能であることが重要なのはなぜですか。

画像分類も文章生成も、この基本的な流れは同じです。ただし、入力の構造と、特徴を抽出するための層が異なります。

---

## 2. CNN（Convolutional Neural Network）

### 2-1. 画像は数値の配列

グレースケール画像は「高さ × 幅 × 1」、カラー画像は通常「高さ × 幅 × 3」のテンソルとして表現されます。MNISTの画像は `28 × 28` ピクセルのグレースケール画像です。

画像を単純な全結合層へ入れるには、すべてのピクセルを1列に並べる必要があります。しかし、それでは「隣り合うピクセル」という画像の空間構造を直接利用しにくくなります。

### 2-2. 畳み込み層

畳み込み層では、小さなフィルタ（カーネル）を画像上で動かしながら、局所的なパターンを検出します。学習開始時のフィルタはランダムですが、学習によって線、角、曲線などを検出する重みに変化します。

#### フィルタはパターンへの「反応の強さ」を計算する

フィルタは、画像の小領域と要素ごとの掛け算を行い、その値を合計して1つの数値を作ります。位置 $(r,c)$ における計算は、バイアスを含めると次のように表せます。

$$
z_{r,c}
= \sum_i \sum_j X_{r+i,c+j}K_{i,j} + b
$$

- $X$：画像のピクセル値
- $K$：フィルタの重み
- $b$：フィルタごとのバイアス
- $z_{r,c}$：その位置でのフィルタの反応

```mermaid
flowchart LR
    patch["画像の小領域<br/>例：3 × 3ピクセル"]:::input
    kernel["フィルタ<br/>3 × 3の重み"]:::kernel
    multiply["対応する要素を<br/>掛け算"]:::process
    sum["合計して<br/>バイアスを加える"]:::process
    relu["ReLU<br/>負の反応を0にする"]:::activation
    feature["特徴マップの<br/>1ピクセル"]:::output

    patch --> multiply
    kernel --> multiply
    multiply --> sum --> relu --> feature

    classDef input fill:#dbeafe,stroke:#2563eb,color:#172554,stroke-width:2px
    classDef kernel fill:#fef3c7,stroke:#d97706,color:#451a03,stroke-width:2px
    classDef process fill:#ede9fe,stroke:#7c3aed,color:#2e1065,stroke-width:2px
    classDef activation fill:#dcfce7,stroke:#16a34a,color:#052e16,stroke-width:2px
    classDef output fill:#fee2e2,stroke:#dc2626,color:#450a0a,stroke-width:2px
```

たとえば、次のフィルタは、左側が暗く右側が明るくなる縦方向の境界に強く反応します。

$$
K =
\begin{bmatrix}
-1 & 0 & 1 \\
-1 & 0 & 1 \\
-1 & 0 & 1
\end{bmatrix}
$$

画像の小領域が次のようになっているとします。

$$
X_{\mathrm{patch}} =
\begin{bmatrix}
0 & 0 & 1 \\
0 & 0 & 1 \\
0 & 0 & 1
\end{bmatrix}
$$

対応する値を掛けて合計すると、バイアスを0とした場合の反応は次のようになります。

$$
z
= (0 \times -1 + 0 \times 0 + 1 \times 1) \times 3
= 3
$$

フィルタが探している向きの境界があるため、大きな正の値が得られます。一方、明暗が逆向きなら反応は$-3$となり、ReLUを通した後は0になります。領域全体が同じ明るさなら、$-1+0+1$が打ち消し合い、反応はほぼ0です。

したがって、「フィルタと完全に一致した場合だけ出力する」という二値的な処理ではありません。

- よく対応するパターン：大きな正の反応
- 部分的に対応するパターン：中程度の反応
- 対応しないパターン：0に近い反応
- 反対向きのパターン：負の反応。ReLU後は0になることが多い

この計算を画像のすべての位置で繰り返して並べたものが**特徴マップ**です。探している線や境界が画像のどこにあるかを、反応の強さを保った新しい画像として表します。

```text
入力画像の各位置で同じフィルタを計算
                    ↓
反応が弱い位置：0に近い値
反応が強い位置：大きな値
                    ↓
位置ごとの反応を並べた特徴マップ
```

実際のCNNではフィルタを人間が指定しません。どのフィルタが分類に役立つかを、バックプロパゲーションによってデータから学習します。また、`Conv2D(32, ...)`は32種類のフィルタを学習し、それぞれに対応する32枚の特徴マップを出力します。

CNNが画像に適している主な理由は次の2点です。

- **局所受容野**：近くのピクセルをまとめて調べる
- **重み共有**：同じフィルタを画像の各位置で使うため、パラメータ数を抑えられる

入力サイズを $H \times W$、カーネルサイズを $K_H \times K_W$、上下・左右のパディングをそれぞれ $P_H, P_W$、縦・横のストライドをそれぞれ $S_H, S_W$ とします。このとき、畳み込み後の高さ $H_{\mathrm{out}}$ と幅 $W_{\mathrm{out}}$ は次式で求められます。

$$
H_{\mathrm{out}}
= \left\lfloor \frac{H + 2P_H - K_H}{S_H} \right\rfloor + 1
$$

$$
W_{\mathrm{out}}
= \left\lfloor \frac{W + 2P_W - K_W}{S_W} \right\rfloor + 1
$$

たとえば、入力が $28 \times 28$、カーネルが $3 \times 3$、パディングが上下左右に1、ストライドが縦横ともに1なら、出力は次のようになります。

$$
H_{\mathrm{out}} = W_{\mathrm{out}}
= \left\lfloor \frac{28 + 2 \times 1 - 3}{1} \right\rfloor + 1
= 28
$$

このように、奇数サイズのカーネルで`padding="same"`、`strides=1`を指定すると、畳み込み前後で高さと幅を保てます。

### 2-3. プーリング層

Max Poolingは小領域の最大値を残して特徴マップを縮小します。

- 計算量を減らす
- 小さな位置ずれの影響を受けにくくする
- 重要な反応を残す

一方で、縮小しすぎると細かな位置情報を失います。

### 2-4. CNNの階層構造

CNNでは、一般に浅い層から深い層へ進むにつれて、特徴が抽象化されます。

```text
画像 → 線・エッジ → 角・曲線 → 部品 → 物体 → クラス確率
```

今回のモデルは次の流れです。

```text
28×28×1
  ↓ Conv2D（32枚の特徴マップ）
28×28×32
  ↓ MaxPooling2D
14×14×32
  ↓ Conv2D（64枚の特徴マップ）
14×14×64
  ↓ MaxPooling2D
 7×7×64
  ↓ Flatten → Dense → Dropout → Dense(10, softmax)
数字0〜9の確率
```

### 2-5. 出力と損失関数

10クラス分類では、出力層に10個のユニットを置き、`softmax` で合計1の確率へ変換します。MNISTの正解ラベルは0〜9の整数なので、損失関数には `sparse_categorical_crossentropy` を使います。

> `categorical_crossentropy` は正解をone-hot表現にした場合に使います。ラベルの形式と損失関数を対応させることが重要です。

---

# 第15回：LLMの基礎とMNIST CNN演習

## 3. LLM（Large Language Model）

### 3-1. 文章を数値に変える

LLMは文章をそのまま理解するわけではありません。おおまかに次の順で処理します。

```text
文章 → トークン分割 → トークンID → 埋め込みベクトル
     → Transformer層 → 次のトークンの確率
```

**トークン**は、文字、単語、単語の一部など、モデルが処理する文章の単位です。同じ文章でも、使用するトークナイザによって分割方法は異なります。

**埋め込み（Embedding）**は、各トークンを多数の数値からなるベクトルへ変換する仕組みです。学習によって、使われ方が似たトークンは関連する表現を持つようになります。

### 3-2. TransformerとSelf-Attention

Transformerの中心にあるSelf-Attentionは、文中の各トークンが他のどのトークンと強く関係するかを計算します。各トークンからQuery、Key、Valueというベクトルを作り、概念的には次の計算を行います。

$$
\mathrm{Attention}(Q,K,V)
= \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

- $QK^\top$：トークン同士の関連度
- $\sqrt{d_k}$：値が極端に大きくなるのを抑える調整
- `softmax`：どこに注目するかを重みに変換
- $V$：重みに応じて集約する情報

複数の注目方法を並列に学習する仕組みを**Multi-Head Attention**と呼びます。また、語順を区別するために位置情報も加えます。

### 3-3. 次トークン予測

生成型LLMの基本的な学習課題は、直前までのトークンから次のトークンを予測することです。

```text
入力：「データサイエンスを学ぶ目的は」
出力候補：「データ」「問題」「分析」…の確率
```

生成時には、選ばれたトークンを入力へ追加し、次のトークンを再び予測します。これを繰り返して文章を作ります。出力は確率に基づくため、流暢でも事実とは限りません。

### 3-4. 学習と利用の段階

- **事前学習**：大量の文章から言語パターンを学ぶ
- **指示調整**：人間の指示に沿って応答しやすくする
- **推論**：入力されたプロンプトから応答を生成する
- **RAG**：検索した外部資料を文脈として与え、回答に利用する
- **ファインチューニング**：特定用途のデータで追加学習する

### 3-5. LLM利用時の注意

- **ハルシネーション**：存在しない事実や出典を生成することがある
- **バイアス**：学習データの偏りが出力へ反映されることがある
- **プロンプト依存性**：指示の表現や文脈で出力が変わる
- **機密性**：個人情報・診療情報・未公開データを安易に入力しない
- **検証責任**：重要な判断では一次資料や専門家による確認が必要

---

## 4. CNNとLLMを比較する

| 観点 | CNN | LLM |
| --- | --- | --- |
| 主な入力 | 画像などの格子状データ | トークン列 |
| 数値表現 | ピクセル値 | トークンIDと埋め込み |
| 中心的な処理 | 畳み込み | Self-Attention |
| 得意な関係 | 近傍の局所パターン | 文脈中の広い関係 |
| 代表的な出力 | クラス、位置、領域 | 次トークン、文章、分類 |
| 共通点 | パラメータ、順伝播、損失、誤差逆伝播、最適化 |

近年は、画像を小領域に分割してTransformerへ入力するVision Transformerや、画像と文章を同時に扱うマルチモーダルモデルもあります。したがって「CNNは画像、Transformerは文章」という区別は絶対ではありません。

### 第14回 確認問題

1. 全結合層と比べて、CNNのパラメータ数を抑えやすい理由は何ですか。
2. Max Poolingの利点と、情報を失うという欠点を説明してください。
3. LLMが文章を生成する基本単位は何ですか。
4. Self-Attentionは何を計算していますか。
5. CNNとLLMに共通する学習の4段階を書いてください。
6. LLMの回答を重要な意思決定にそのまま使ってはいけない理由を2つ挙げてください。

---

## 5. 実行環境の準備

Google Colabでは、必要に応じて「ランタイム」→「ランタイムのタイプを変更」からGPUを選択できます。MNISTはCPUでも実行できます。


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras

from keras import layers
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

sns.set_theme(style="whitegrid")
print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 6. MNISTデータを読み込む

MNISTには、0〜9の手書き数字画像が学習用60,000枚、テスト用10,000枚含まれます。


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test :", x_test.shape, x_test.dtype)
print("y_test :", y_test.shape, y_test.dtype)
print("pixel range:", x_train.min(), "〜", x_train.max())


### 演習1：データを観察する

次のセルを実行し、画像とラベルが対応していることを確認します。


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(10, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(f"label = {y_train[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


クラス数も確認します。


In [ ]:
class_counts = pd.Series(y_train).value_counts().sort_index()
display(class_counts.rename("count").to_frame())

class_counts.plot(kind="bar", figsize=(8, 4), color="steelblue")
plt.xlabel("digit")
plt.ylabel("count")
plt.title("MNIST training class distribution")
plt.show()


**考察**

- 画像の形状 `(28, 28)` は何を表していますか。
- 数字ごとの枚数は完全に同じですか。
- 背景と文字は、それぞれどのようなピクセル値ですか。

## 7. 前処理

ピクセル値を0〜255から0〜1へ変換し、チャンネル次元を追加します。


In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("x_train:", x_train.shape)
print("x_test :", x_test.shape)
print("pixel range:", x_train.min(), "〜", x_train.max())


> 形状の最後の `1` はグレースケールの1チャンネルを表します。RGB画像なら通常は `3` です。

### 演習2：前処理を説明する

次の問いに文章で答えてください。

1. 255で割る目的は何ですか。
2. `(60000, 28, 28)` を `(60000, 28, 28, 1)` に変えた理由は何ですか。
3. `y_train` をone-hot表現に変換していないのに使える損失関数は何ですか。

## 8. CNNモデルを構築する

### 8-1. Dropoutとdrop rate

ニューラルネットワークのユニット数や層数が増えると、学習データにだけ細かく適合し、未知データへの性能が下がる**過学習**が起きることがあります。Dropoutは、過学習を抑えるための正則化手法の1つです。

学習中に、層の出力の一部をランダムに0へ置き換えます。毎回異なるユニットが無効になるため、モデルは特定のユニットだけに頼りにくくなります。

```mermaid
flowchart LR
    dense["Dense層<br/>64ユニット"]:::layer
    dropout{"Dropout<br/>rate = 0.3"}:::dropout
    active["約70%を使用<br/>次の層へ伝える"]:::active
    inactive["約30%を<br/>一時的に0にする"]:::inactive
    output["出力層"]:::layer

    dense --> dropout
    dropout --> active --> output
    dropout -.-> inactive

    classDef layer fill:#dbeafe,stroke:#2563eb,color:#172554,stroke-width:2px
    classDef dropout fill:#fef3c7,stroke:#d97706,color:#451a03,stroke-width:2px
    classDef active fill:#dcfce7,stroke:#16a34a,color:#052e16,stroke-width:2px
    classDef inactive fill:#f3f4f6,stroke:#6b7280,color:#111827,stroke-width:2px,stroke-dasharray:5 5
```

`layers.Dropout(0.3)`の`0.3`が**drop rate**です。

| 項目 | `Dropout(0.3)`での意味 |
| --- | --- |
| 無効にする割合 | 30% |
| 残す割合 | 70% |
| 64ユニット中、1回の処理で残る数の期待値 | $64 \times 0.7 = 44.8$個 |

Dropoutの動作は学習時と評価・予測時で異なります。

| 状態 | Dropoutの動作 |
| --- | --- |
| 学習時：`model.fit()` | 指定割合をランダムに無効化する |
| 評価時：`model.evaluate()` | すべてのユニットを使用する |
| 予測時：`model.predict()` | すべてのユニットを使用する |

Kerasは、学習時と評価・予測時の切り替えや出力スケールの調整を自動的に行います。drop rateを大きくしすぎると必要な情報まで失われ、学習不足になる可能性があります。今回の`0.3`は固定的な正解ではなく、検証データを見ながら調整するハイパーパラメータです。

### 8-2. モデルの定義

ここから先は、必要な処理を自分で整理し、AIにコード作成を依頼しながら進めます。AIが生成したコードをそのまま実行するのではなく、指定した条件が反映されているか、前のセルで作成した変数とつながっているかを確認してください。

#### 作成するモデル

次の順に層を配置します。

| 順番 | 層 | 設定 | 目的 |
| ---: | --- | --- | --- |
| 1 | Input | `(28, 28, 1)` | MNIST画像を受け取る |
| 2 | Conv2D | 32フィルタ、3×3、same、ReLU、名前は`conv1` | 局所特徴を抽出する |
| 3 | MaxPooling2D | 2×2 | 特徴マップを縮小する |
| 4 | Conv2D | 64フィルタ、3×3、same、ReLU | より複雑な特徴を抽出する |
| 5 | MaxPooling2D | 2×2 | 特徴マップを縮小する |
| 6 | Flatten |  | 全結合層へ渡せる形にする |
| 7 | Dense | 64ユニット、ReLU | 分類に必要な特徴を統合する |
| 8 | Dropout | drop rate 0.3 | 過学習を抑える |
| 9 | Dense | 10ユニット、Softmax | 0〜9の確率を出力する |

モデル名は`mnist_cnn`、変数名は`model`とします。作成後、Adam、`sparse_categorical_crossentropy`、正解率を指定してコンパイルし、`summary()`を表示します。

**AIへの依頼例**

```text
Keras 3のSequential APIを使って、MNIST分類用CNNを作成してください。
入力は28×28×1です。以下の順に層を配置してください。
1. 32枚の3×3 Conv2D。paddingはsame、活性化関数はReLU、層名はconv1
2. 2×2 MaxPooling2D
3. 64枚の3×3 Conv2D。paddingはsame、活性化関数はReLU
4. 2×2 MaxPooling2D
5. Flatten
6. 64ユニットのDense。活性化関数はReLU
7. drop rate 0.3のDropout
8. 10ユニットのDense。活性化関数はSoftmax
モデルの変数名はmodel、モデル名はmnist_cnnにしてください。
Adam、sparse_categorical_crossentropy、accuracyでコンパイルし、summaryも表示してください。
すでにimport kerasとfrom keras import layersは実行済みです。
```


In [ ]:
# AIが作成したCNNの定義、コンパイル、summary表示をここに記入する


#### AIが作ったコードの確認

- 入力形状の最後にチャンネル数`1`があるか
- 2つの畳み込み層のフィルタ数が32、64になっているか
- 隠れ層がReLU、出力層がSoftmaxになっているか
- ラベル形式に対応した損失関数になっているか
- 最初の畳み込み層に`conv1`という名前が付いているか

### 演習3：モデルの形状とパラメータ

`model.summary()`を見て答えてください。

1. 1回目のMax Pooling後の形状はいくつですか。
2. 2回目のMax Pooling後の形状はいくつですか。
3. 最初のConv2D層のパラメータ数を、次式で確認してください。

$$
(\text{カーネル高} \times \text{カーネル幅} \times \text{入力チャンネル数} + 1) \times \text{フィルタ数}
$$

4. 最後のDense層のユニット数が10である理由を説明してください。
5. `Dropout(0.3)`は学習時に何%を無効化し、何%を残しますか。
6. Dropoutを評価・予測時にもランダムに適用すると、どのような問題が起こるか考えてください。

## 9. モデルを学習する

学習データの20%を検証データとして分けます。テストデータは、モデルの最終評価まで使用しません。

実装する条件は次のとおりです。

- エポック数：5
- バッチサイズ：128
- 検証データ：学習データの20%
- EarlyStoppingの監視対象：検証損失
- EarlyStoppingの`patience`：2
- 最も良かった重みを復元する
- 学習履歴の変数名：`history`
- テストデータはまだ使わない

**AIへの依頼例**

```text
コンパイル済みのKerasモデルmodelを学習するコードを書いてください。
x_trainとy_trainを使用し、エポック数5、バッチサイズ128、validation_split=0.2とします。
検証損失を監視するEarlyStoppingを設定し、patience=2、restore_best_weights=Trueにしてください。
学習履歴はhistoryへ保存してください。テストデータは使用しないでください。
```


In [ ]:
# AIが作成したEarlyStoppingとモデル学習のコードをここに記入する


> `x_test`や`y_test`が`fit()`の中に入っていた場合は修正してください。テストデータを見ながらモデルを調整すると、最終評価が楽観的になります。

### 学習曲線を確認する

`history.history`には、エポックごとの学習損失、検証損失、学習正解率、検証正解率が記録されています。これをDataFrameに変換し、損失と正解率を左右に並べて可視化します。

**AIへの依頼例**

```text
Kerasの学習履歴historyについて、history.historyをpandasのDataFrameへ変換し、表として表示してください。
次にmatplotlibで1行2列のグラフを作ってください。
左は学習損失lossと検証損失val_loss、右は学習正解率accuracyと検証正解率val_accuracyです。
横軸は1から始まるepochとし、凡例、軸ラベル、タイトルを付けてください。
```


In [ ]:
# AIが作成した学習履歴の表と学習曲線のコードをここに記入する


### 演習4：学習曲線を読む

- エポックが進むと、学習・検証の損失はどう変化しましたか。
- 学習精度だけが上がり、検証精度が改善しなくなる現象を何と呼びますか。
- `Dropout` と `EarlyStopping` はそれぞれ何のために使っていますか。

## 10. テストデータで最終評価する

学習とモデル選択が終わったので、ここで初めてテストデータを使います。`model.evaluate()`でテスト損失とテスト正解率を計算し、小数第4位まで表示します。変数名は`test_loss`と`test_accuracy`にします。

**AIへの依頼例**

```text
学習済みKerasモデルmodelをx_testとy_testで評価してください。
返された損失をtest_loss、正解率をtest_accuracyへ保存し、小数第4位まで表示するコードを書いてください。
```


In [ ]:
# AIが作成したテストデータでの最終評価コードをここに記入する


正解率だけでは、どの数字を間違えやすいか分かりません。予測クラスを求め、混同行列とクラス別指標を確認します。

次に、各画像について10クラスの予測確率を求め、最も確率が高い数字を予測クラスとします。その後、混同行列をヒートマップで表示し、クラス別のprecision、recall、F1-scoreを確認します。

使用する変数名：

- `y_prob`：10クラスの予測確率
- `y_pred`：予測クラス
- `cm`：混同行列

**AIへの依頼例**

```text
学習済みKerasモデルmodelでx_testの予測確率を計算してy_probへ保存してください。
各行で最も確率が高いクラスをNumPyで求め、y_predへ保存してください。
y_testとy_predから混同行列cmを作り、seabornのheatmapで数字も表示してください。
横軸をpredicted label、縦軸をtrue labelとしてください。
最後にscikit-learnのclassification_reportを小数第4位まで表示してください。
必要なライブラリはすでに読み込み済みです。
```


In [ ]:
# AIが作成した予測、混同行列、クラス別評価のコードをここに記入する


#### AIが作ったコードの確認

- `argmax`を取る軸がクラス方向になっているか
- 混同行列で正解と予測の順番が逆になっていないか
- 混同行列の縦軸と横軸の表示が計算内容と一致しているか

### 演習5：評価結果を読む

1. テスト正解率はいくつでしたか。
2. 混同行列の対角成分は何を表しますか。
3. 最も多い誤分類の「正解→予測」の組み合わせを調べてください。

対角成分は正しく分類した数なので、コピーした混同行列の対角成分を0にしてから最大値の位置を求めます。元の`cm`は後でも使用するため、直接書き換えないでください。

**AIへの依頼例**

```text
NumPy配列cmは、行が正解ラベル、列が予測ラベルの混同行列です。
cmをコピーし、コピーした配列の対角成分を0にして、対角成分以外で最大となる行番号、列番号、件数を求めて表示してください。
元のcmは変更しないでください。
```


In [ ]:
# AIが作成した「最も多い誤分類」を求めるコードをここに記入する


## 11. 誤分類画像を観察する

誤分類された画像のインデックスを求め、先頭15枚を3行5列で表示します。各画像には正解、予測、予測クラスに対する確信度を表示します。

**AIへの依頼例**

```text
y_testとy_predが異なる要素のインデックスをNumPyで求めてwrong_indicesへ保存してください。
誤分類画像の先頭15枚を、matplotlibで3行5列に表示してください。
画像はx_testから取り出し、グレースケールで表示します。
各画像のタイトルに正解ラベル、予測ラベル、モデルが予測クラスへ与えた確率を小数第2位まで表示してください。
予測確率はy_probに保存されています。軸は非表示にしてください。
```


In [ ]:
# AIが作成した誤分類画像の可視化コードをここに記入する


### 演習6：誤分類を考察する

誤分類画像を3枚以上選び、次の観点で考察してください。

- 人間にとっても判別しにくい書き方か
- どの数字の特徴と似ているか
- モデルの確信度は高いか低いか
- 画像のずれ、線の太さ、欠けなどが影響していそうか

## 12. 畳み込み層の特徴マップを見る

1枚の画像が、最初の畳み込み層によってどのような特徴マップへ変換されるかを確認します。

モデル作成時に最初の畳み込み層へ`conv1`という名前を付けました。この層の出力を取り出す中間モデルを作成し、テスト画像1枚から得られる32枚の特徴マップを4行8列で表示します。

**AIへの依頼例**

```text
学習済みKerasモデルmodelから、名前がconv1の層の出力を取得する中間モデルconv1_modelを作ってください。
x_testの0番目の画像1枚を入力し、特徴マップをfeature_mapsへ保存してください。
conv1には32フィルタがあるので、32枚の特徴マップをmatplotlibで4行8列に表示してください。
各図にはチャンネル番号を付け、軸は非表示にしてください。
全体タイトルには元画像の正解ラベルも表示してください。
```


In [ ]:
# AIが作成した中間モデルと特徴マップの可視化コードをここに記入する


#### AIが作ったコードの確認

- 中間モデルへの入力が元の`model.inputs`になっているか
- 出力が最終分類結果ではなく`conv1`層の出力になっているか
- 画像1枚でもバッチ次元を残して入力しているか
- 特徴マップのチャンネル方向を32枚に分けて表示しているか

> 特徴マップの意味は、1枚だけを見て断定できません。「縦線に反応しているように見える」など、観察に基づく表現を使いましょう。

## 13. 発展課題：モデルを改善して比較する

基準モデルから**1項目だけ**変更し、同じ条件で再学習してください。複数項目を同時に変えると、どの変更が結果に影響したか判断しにくくなります。

変更候補：

- 1層目のフィルタ数：`32 → 16` または `64`
- カーネルサイズ：`3 → 5`
- Dropout率：`0.3 → 0.5`
- Dense層：`64 → 128`
- Batch size：`128 → 64` または `256`
- データ拡張：小さな回転や平行移動を追加

変更する項目を1つ決めてから、AIへ基準モデルの条件と変更点を伝えます。変数名は`improved_model`とし、基準モデルと同じデータ分割、エポック数、評価方法を使用してください。

**AIへの依頼例**

```text
先ほど作成したMNIST用CNNを基準に、Dropout率だけを0.3から0.5へ変更したKerasモデルを作ってください。
変数名はimproved_modelとしてください。他の層構成、活性化関数、optimizer、loss、metricsは基準モデルと同じにしてください。
基準モデルと同じ条件で学習・評価し、パラメータ数、最高検証精度、テスト精度を取得してください。
```


In [ ]:
# 変更点を1つ決め、AIが作成した改良モデルの定義・学習・評価を記入する


#### 比較時の注意

- 基準モデルから変更した項目を明記する
- テストデータを学習やEarlyStoppingに使わない
- 乱数シード、検証割合、エポック数など、変更対象以外の条件を揃える
- 1回の結果だけで「必ず改善する」と結論付けない

結果を次の表へ記録してください。

| モデル | 変更点 | パラメータ数 | 最高検証精度 | テスト精度 | 考察 |
| --- | --- | ---: | ---: | ---: | --- |
| 基準CNN | なし |  |  |  |  |
| 改良CNN |  |  |  |  |  |

---

> 実行のたびに値がわずかに変わることがあります。乱数シードを固定しても、GPUなどの実行環境によって完全には一致しない場合があります。

---

## 14. まとめ

- CNNは、畳み込みによって画像の局所的な特徴を階層的に学習する
- LLMは、トークンとSelf-Attentionを用いて文脈を処理し、次のトークンを予測する
- 両者は入力と中心的な処理が異なるが、損失を小さくするようにパラメータを学習する点は共通する
- モデル評価では、正解率だけでなく学習曲線、混同行列、個別の誤分類例も確認する
- 高い性能が得られても、未知データへの一般化、バイアス、説明可能性、利用目的を検討する必要がある

---

**Last updated**: 2026-07-20  
**Instructor**: Nakaura-T (DS Department, Kumamoto Univ)
